# KAGGLE_B02_D_C0_seed42

Purpose: locked final classification run for `D-C0`, `DenseNet121`, seed `42`.

Add these Kaggle datasets:

- `hintrngia/gate46-development-only-cxr`
- `hintrngia/gate7-test-seal`
- `hintrngia/cxr-lung-masks`

Outputs are written to:

`/kaggle/working/locked_outputs/classification_runs/D-C0/DenseNet121/seed_42/`

This notebook extracts temporary image data into `/tmp/xai_locked_runtime/` so Kaggle output does not include thousands of source images.


In [ ]:
!pip install -q albumentations grad-cam


## Embedded Module: 02_config.py


In [ ]:
import dataclasses
from typing import List, Tuple

@dataclasses.dataclass
class PipelineConfig:
    # Environment & Seeds
    seeds: List[int] = dataclasses.field(default_factory=lambda: [3407, 42, 2024])
    
    # Dataset & Dataloader
    batch_size: int = 32
    num_workers: int = 2
    input_resolution: int = 256  # configurable to 224, 320, 384
    
    # Model configuration
    models_to_train: List[str] = dataclasses.field(default_factory=lambda: ["DenseNet121", "ResNet50"])
    densenet_freeze_percent: float = 0.75
    resnet_freeze_percent: float = 0.80
    
    # Early Stopping & Checkpoints
    patience: int = 5
    min_delta: float = 1e-4
    skip_training_if_checkpoint_exists: bool = True  # Automatically loads existing .pt checkpoints to skip training and execute XAI directly
    
    # Training (Stage 1 - Head Only)
    stage1_epochs: int = 5
    stage1_lr: float = 1e-3
    
    # Training (Stage 2 - Fine tuning)
    stage2_epochs: int = 25
    stage2_lr: float = 1e-4
    stage2_min_lr: float = 1e-6
    weight_decay: float = 1e-4
    
    # Optimization
    mixed_precision: bool = True
    gradient_accumulation_steps: int = 1
    gradient_clipping_max_norm: float = 1.0
    
    # Data Augmentation (Albumentations)
    apply_clahe: bool = True
    
    # Thresholding & TTA
    use_tta: bool = True
    
    # XAI
    explainability_methods: List[str] = dataclasses.field(default_factory=lambda: ["Grad-CAM", "Guided Grad-CAM"])

    # Loss Configuration
    loss_type: str = "Focal" # Options: "BCE", "Focal"
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0
    
    # Architecture Enhancements
    use_cbam: bool = True
    
    # Auxiliary Loss (Mask-Guided Attention)
    use_mask_loss: bool = True
    mask_loss_weight: float = 0.1
    
# Global Configuration instance
CONFIG = PipelineConfig()


## Embedded Module: 04_dataset.py


In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2

class ChestXRayDataset(Dataset):
    def __init__(self, df, image_map: dict, config, split: str = 'train', mask_dir: Path = None):
        """
        df: DataFrame containing dataset info.
        image_map: Dictionary mapping filename to absolute Path.
        split: 'train', 'val', or 'test'
        mask_dir: Optional path to lung masks directory.
        """
        self.df = df
        self.image_map = image_map
        self.config = config
        self.split = split
        self.mask_dir = mask_dir
        
        # Determine image path column
        if 'archive_member' in df.columns:
            self.image_filenames = df['archive_member'].apply(lambda x: Path(x).name).values
        elif 'source_relative_path' in df.columns:
            self.image_filenames = df['source_relative_path'].apply(lambda x: Path(x).name).values
        elif 'image' in df.columns:
            self.image_filenames = df['image'].apply(lambda x: Path(x).name).values
        else:
            raise KeyError("DataFrame must contain 'archive_member', 'source_relative_path', or 'image' column")
            
        # Determine label column
        if 'label_index' in df.columns:
            self.labels = df['label_index'].values
        elif 'label' in df.columns:
            if df['label'].dtype == object:
                self.labels = (df['label'].str.strip().str.upper() == 'PNEUMONIA').astype(int).values
            else:
                self.labels = df['label'].values
        else:
            self.labels = np.zeros(len(df)) # Fallback if no labels
        
        self.transform = self._get_transforms()
        
        # CLAHE initialization
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        
    def _get_transforms(self):
        res = self.config.input_resolution
        
        if self.split == 'train':
            return A.Compose([
                A.Resize(res, res),
                A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.RandomGamma(p=0.3),
                A.GaussNoise(p=0.2),
                A.MotionBlur(p=0.2),
                A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, res//10), hole_width_range=(1, res//10), p=0.2),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ])
        else:
            return A.Compose([
                A.Resize(res, res),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        
        if img_name not in self.image_map:
            raise FileNotFoundError(f"Image {img_name} not found in the provided dataset folders.")
            
        img_path = self.image_map[img_name]
        
        # Branch A: Raw Image Reading (Grayscale to match X-ray origin, then to RGB once)
        image = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"OpenCV could not read {img_path}")
            
        # Branch B: CLAHE Preprocessing
        if self.config.apply_clahe:
            image = self.clahe.apply(image)
            
        # Convert to RGB exactly once
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        # Load mask if needed
        mask = None
        if self.config.use_mask_loss and self.mask_dir is not None:
            mask_path = self.mask_dir / img_name
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                mask = (mask > 0).astype(np.float32)
            else:
                mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)
        
        # Apply Albumentations
        if mask is not None:
            augmented = self.transform(image=image, mask=mask)
            image_tensor = augmented['image']
            mask_tensor = augmented['mask'].unsqueeze(0) # [1, H, W]
        else:
            augmented = self.transform(image=image)
            image_tensor = augmented['image']
            mask_tensor = torch.zeros((1, image_tensor.shape[1], image_tensor.shape[2]))
        
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return image_tensor, label, img_name, mask_tensor

def get_class_weights(df):
    """Calculates class weights for imbalanced datasets."""
    # Support both label_index (int) and label (string) columns
    if 'label_index' in df.columns:
        label_series = df['label_index'].astype(int)
    elif 'label' in df.columns:
        if df['label'].dtype == object:
            label_series = (df['label'].str.strip().str.upper() == 'PNEUMONIA').astype(int)
        else:
            label_series = df['label'].astype(int)
    else:
        return torch.tensor([1.0], dtype=torch.float32)
    
    pos = (label_series == 1).sum()
    neg = (label_series == 0).sum()
    total = pos + neg
    if pos == 0 or neg == 0:
        return torch.tensor([1.0], dtype=torch.float32)
    weight_for_1 = (1 / pos) * (total / 2.0)
    return torch.tensor([weight_for_1], dtype=torch.float32)  # pos_weight for BCEWithLogitsLoss


## Embedded Module: 05_dataloader.py


In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

def create_dataloaders(train_ds, val_ds, test_ds, config, use_sampler=False):
    """
    Creates DataLoaders with optimized settings for performance.
    """
    # Optional: WeightedRandomSampler instead of focal loss
    train_sampler = None
    if use_sampler and hasattr(train_ds, 'labels'):
        labels = train_ds.labels
        class_sample_count = np.array([len(np.where(labels == t)[0]) for t in np.unique(labels)])
        weight = 1. / class_sample_count
        samples_weight = np.array([weight[int(t)] for t in labels])
        samples_weight = torch.from_numpy(samples_weight).double()
        train_sampler = WeightedRandomSampler(samples_weight, len(samples_weight))
        shuffle = False
    else:
        shuffle = True
        
    import os
    use_pin_memory = torch.cuda.is_available()
    num_workers = 0 if os.name == 'nt' else config.num_workers
    
    train_loader = DataLoader(
        train_ds, 
        batch_size=config.batch_size, 
        shuffle=shuffle,
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
        drop_last=True,
        persistent_workers=True if num_workers > 0 else False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=config.batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
        persistent_workers=True if num_workers > 0 else False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        test_ds, 
        batch_size=config.batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
        persistent_workers=True if num_workers > 0 else False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    return train_loader, val_loader, test_loader


## Embedded Module: 06_models.py


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
           
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv1(x_cat)
        attention_map = self.sigmoid(out)
        return attention_map

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)
        
        # Identity node specifically designed for Grad-CAM to attach a hook
        self.output_node = nn.Identity()

    def forward(self, x):
        x_out = x * self.ca(x)
        spatial_att = self.sa(x_out)
        x_out = x_out * spatial_att
        
        # Pass through identity node for Grad-CAM extraction
        return self.output_node(x_out), spatial_att

class ClassificationModel(nn.Module):
    def __init__(self, architecture: str, freeze_percent: float, use_cbam: bool = False):
        super().__init__()
        self.architecture = architecture
        self.use_cbam = use_cbam
        
        if architecture == "DenseNet121":
            # Load pretrained weights
            self.backbone = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
            in_features = self.backbone.classifier.in_features
            
            if self.use_cbam:
                self.cbam = CBAM(in_features)
                
            # Replace classifier head for binary classification
            self.backbone.classifier = nn.Sequential(
                nn.Linear(in_features, 1)
            )
            self._freeze_layers_densenet(freeze_percent)
            
        elif architecture == "ResNet50":
            # Load pretrained weights
            self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
            in_features = self.backbone.fc.in_features
            
            if self.use_cbam:
                self.cbam = CBAM(in_features)
                
            # Replace classifier head
            self.backbone.fc = nn.Sequential(
                nn.Linear(in_features, 1)
            )
            self._freeze_layers_resnet(freeze_percent)
        else:
            raise ValueError(f"Unknown architecture: {architecture}")

    def forward(self, x, return_attention=False):
        if self.architecture == "DenseNet121":
            features = self.backbone.features(x)
            features = F.relu(features, inplace=True)
            
            spatial_att = None
            if self.use_cbam:
                features, spatial_att = self.cbam(features)
                
            out = F.adaptive_avg_pool2d(features, (1, 1))
            out = torch.flatten(out, 1)
            logits = self.backbone.classifier(out)
            
        elif self.architecture == "ResNet50":
            x = self.backbone.conv1(x)
            x = self.backbone.bn1(x)
            x = self.backbone.relu(x)
            x = self.backbone.maxpool(x)

            x = self.backbone.layer1(x)
            x = self.backbone.layer2(x)
            x = self.backbone.layer3(x)
            features = self.backbone.layer4(x)
            
            spatial_att = None
            if self.use_cbam:
                features, spatial_att = self.cbam(features)
                
            out = self.backbone.avgpool(features)
            out = torch.flatten(out, 1)
            logits = self.backbone.fc(out)
            
        if return_attention:
            return logits, spatial_att
        return logits
        
    def _freeze_layers_densenet(self, freeze_percent):
        """Freezes the specified percentage of layers from the bottom up."""
        total_layers = len(list(self.backbone.features.children()))
        freeze_up_to = int(total_layers * freeze_percent)
        
        for i, child in enumerate(self.backbone.features.children()):
            if i < freeze_up_to:
                for param in child.parameters():
                    param.requires_grad = False
                    
    def _freeze_layers_resnet(self, freeze_percent):
        """Freezes the specified percentage of layers from the bottom up for ResNet."""
        layers = [
            self.backbone.conv1, 
            self.backbone.bn1, 
            self.backbone.relu, 
            self.backbone.maxpool,
            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4
        ]
        
        total_layers = len(layers)
        freeze_up_to = int(total_layers * freeze_percent)
        
        for i, layer in enumerate(layers):
            if i < freeze_up_to:
                for param in layer.parameters():
                    param.requires_grad = False

    def unfreeze_all(self):
        """Unfreezes all layers for Stage 2 training."""
        for param in self.parameters():
            param.requires_grad = True


## Embedded Module: 07_training.py


In [ ]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F
import numpy as np
import time
import copy

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        
        # Calculate alpha properly for positive and negative classes
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

def get_criterion(config, class_weights=None):
    if config.loss_type == "Focal":
        # If class weights are provided, use them as alpha (or a scaled version)
        alpha = config.focal_alpha
        if class_weights is not None:
            # simple mapping of pos_weight to focal alpha
            alpha = float(class_weights[0]) / (1.0 + float(class_weights[0]))
        return FocalLoss(alpha=alpha, gamma=config.focal_gamma)
    else:
        return nn.BCEWithLogitsLoss(pos_weight=class_weights)

def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device, config, current_mask_weight=0.0):
    model.train()
    
    # CRITICAL FIX: Ensure frozen BatchNorm layers stay in eval mode to prevent running stats corruption
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and hasattr(m, 'weight') and m.weight is not None and not m.weight.requires_grad:
            m.eval()
            
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for i, (images, labels, _, masks) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)
        masks = masks.to(device)
        
        use_autocast = (getattr(config, 'mixed_precision', False) and torch.cuda.is_available())
        device_type = "cuda" if torch.cuda.is_available() else "cpu"
        
        with autocast(device_type=device_type, enabled=use_autocast):
            if config.use_mask_loss:
                outputs, spatial_att = model(images, return_attention=True)
                loss_clf = criterion(outputs, labels)
                
                # Resize spatial attention to mask size
                spatial_att = F.interpolate(spatial_att, size=masks.shape[-2:], mode='bilinear', align_corners=False)
                
                # Mask guided loss: encourage attention inside the mask, penalize outside
                # simple BCE between spatial attention and lung mask
                with torch.amp.autocast(device_type, enabled=False):
                    loss_mask = F.binary_cross_entropy(torch.clamp(spatial_att.float(), 1e-7, 1.0 - 1e-7), torch.clamp(masks.float(), 0.0, 1.0))
                
                loss = loss_clf + current_mask_weight * loss_mask
            else:
                outputs = model(images, return_attention=False)
                loss = criterion(outputs, labels)
            
            # Normalize loss for gradient accumulation
            if config.gradient_accumulation_steps > 1:
                loss = loss / config.gradient_accumulation_steps
                
        scaler.scale(loss).backward()
        
        if (i + 1) % config.gradient_accumulation_steps == 0 or (i + 1) == len(dataloader):
            if config.gradient_clipping_max_norm > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clipping_max_norm)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            
        running_loss += loss.item() * config.gradient_accumulation_steps
        
        probs = torch.sigmoid(outputs).detach().cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        
    return running_loss / len(dataloader), np.array(all_preds), np.array(all_labels)

def validate(model, dataloader, criterion, device, config, current_mask_weight=0.0):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    use_autocast = (getattr(config, 'mixed_precision', False) and torch.cuda.is_available())
    device_type = "cuda" if torch.cuda.is_available() else "cpu"
    
    with torch.no_grad():
        for images, labels, _, masks in dataloader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            masks = masks.to(device)
            
            with autocast(device_type=device_type, enabled=use_autocast):
                if config.use_mask_loss:
                    outputs, spatial_att = model(images, return_attention=True)
                    loss_clf = criterion(outputs, labels)
                    spatial_att = F.interpolate(spatial_att, size=masks.shape[-2:], mode='bilinear', align_corners=False)
                    with torch.amp.autocast(device_type, enabled=False):
                        loss_mask = F.binary_cross_entropy(torch.clamp(spatial_att.float(), 1e-7, 1.0 - 1e-7), torch.clamp(masks.float(), 0.0, 1.0))
                    loss = loss_clf + current_mask_weight * loss_mask
                else:
                    outputs = model(images, return_attention=False)
                    loss = criterion(outputs, labels)
                
            running_loss += loss.item()
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            
    return running_loss / len(dataloader), np.array(all_preds), np.array(all_labels)

def train_pipeline(model, train_loader, val_loader, device, config, seed, class_weights=None):
    """
    Executes the Two-Stage transfer learning pipeline.
    """
    if getattr(config, 'skip_training_if_checkpoint_exists', False):
        import os
        from pathlib import Path
        model_name = f"{model.architecture}_seed{seed}_best.pt"
        
        # Search candidate directories
        candidate_dirs = [
            Path("/content/drive/MyDrive/q1_rebuild_outputs/models"),
            Path("/content/q1_rebuild_outputs/models"),
            Path("/content/models"),
            Path("/content"),
            Path("./outputs/models"),
            Path("./local_work/models"),
            Path("./models"),
            Path("."),
            Path("/kaggle/working/models"),
            Path("/kaggle/working")
        ]
        
        chkpt_path = None
        for cdir in candidate_dirs:
            if (cdir / model_name).exists():
                chkpt_path = cdir / model_name
                break
        
        if chkpt_path is None:
            # Fallback recursive search in /content and current dir
            for search_root in [Path("/content/drive/MyDrive"), Path("/content"), Path(".")]:
                if search_root.exists():
                    matches = list(search_root.rglob(model_name))
                    if matches:
                        chkpt_path = matches[0]
                        break
        
        if chkpt_path is not None and chkpt_path.exists():
            print(f"\n[TRAIN] Found existing checkpoint: {chkpt_path}")
            print("[TRAIN] skip_training_if_checkpoint_exists=True. Loading weights and skipping training...")
            model.load_state_dict(torch.load(chkpt_path, map_location=device, weights_only=True))
            model.to(device)
            return model

    print(f"\n--- Starting Training for {model.architecture} (Seed: {seed}) ---")
    model = model.to(device)
    
    if class_weights is not None:
        class_weights = class_weights.to(device)
        
    criterion = get_criterion(config, class_weights)
    scaler = GradScaler(device="cuda", enabled=config.mixed_precision)
    
    # -----------------------
    # STAGE 1: Head Only
    # -----------------------
    print("\n[STAGE 1] Training Classifier Head Only")
    optimizer_stage1 = AdamW(filter(lambda p: p.requires_grad, model.parameters()), 
                             lr=config.stage1_lr, weight_decay=config.weight_decay)
    
    for epoch in range(config.stage1_epochs):
        # Stage 1: Warm-up mask loss while backbone is frozen to safely train CBAM
        current_mask_weight = config.mask_loss_weight * ((epoch + 1) / config.stage1_epochs)
        
        train_loss, _, _ = train_one_epoch(model, train_loader, optimizer_stage1, criterion, scaler, device, config, current_mask_weight)
        val_loss, _, _ = validate(model, val_loader, criterion, device, config, current_mask_weight)
        print(f"Stage 1 - Epoch {epoch+1}/{config.stage1_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Mask W: {current_mask_weight:.2f}")
        
    # -----------------------
    # STAGE 2: Fine-Tuning
    # -----------------------
    print("\n[STAGE 2] Unfreezing Backbone for Fine-Tuning")
    model.unfreeze_all()
    optimizer_stage2 = AdamW(model.parameters(), lr=config.stage2_lr, weight_decay=config.weight_decay)
    scheduler = CosineAnnealingLR(optimizer_stage2, T_max=config.stage2_epochs, eta_min=config.stage2_min_lr)
    
    best_val_loss = float('inf')
    best_weights = None
    patience_counter = 0
    
    for epoch in range(config.stage2_epochs):
        # Stage 2: Mask loss is fully active
        current_mask_weight = config.mask_loss_weight
        
        train_loss, _, _ = train_one_epoch(model, train_loader, optimizer_stage2, criterion, scaler, device, config, current_mask_weight)
        val_loss, val_preds, val_labels = validate(model, val_loader, criterion, device, config, current_mask_weight)
        
        scheduler.step()
        
        print(f"Stage 2 - Epoch {epoch+1}/{config.stage2_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | Mask W: {current_mask_weight:.3f}")
        
        if val_loss < best_val_loss - config.min_delta:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print("  [*] Saved new best model!")
        else:
            patience_counter += 1
            print(f"  [!] Early stopping counter: {patience_counter} / {config.patience}")
            if patience_counter >= config.patience:
                print(f"  [!] Early stopping triggered. Training stopped at epoch {epoch+1}.")
                break
            
    # Restore best weights
    model.load_state_dict(best_weights)
    print("Training Complete. Restored best validation weights.")
    return model


## Embedded Module: 08_validation.py


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import brier_score_loss, log_loss

def calculate_metrics(labels, probs, threshold=0.5):
    """
    Computes all standard evaluation metrics required by the Gate 7 protocol.
    """
    preds = (probs >= threshold).astype(int)
    
    auc = roc_auc_score(labels, probs)
    auprc = average_precision_score(labels, probs)
    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0) # Sensitivity
    f1 = f1_score(labels, preds, zero_division=0)
    
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    balanced_acc = (recall + specificity) / 2
    
    brier = brier_score_loss(labels, probs)
    logloss = log_loss(labels, probs)
    
    # Matthews Correlation Coefficient
    mcc_num = (tp * tn) - (fp * fn)
    mcc_den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = mcc_num / mcc_den if mcc_den > 0 else 0
    
    metrics = {
        "auroc": float(auc),
        "auprc": float(auprc),
        "accuracy": float(acc),
        "precision": float(precision),
        "sensitivity": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_acc),
        "brier_score": float(brier),
        "log_loss": float(logloss),
        "mcc": float(mcc),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "threshold_used": float(threshold)
    }
    return metrics


## Embedded Module: 09_calibration.py


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_curve, f1_score
from scipy.optimize import minimize

class TemperatureScaling(nn.Module):
    """
    Applies temperature scaling on model logits to calibrate probabilities.
    """
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature
        
def find_optimal_threshold(labels, probs, metric='youden'):
    """
    Finds the optimal classification threshold using ROC Curve.
    Metric can be 'youden' or 'f1'.
    """
    fpr, tpr, thresholds = roc_curve(labels, probs)
    
    # Youden's J statistic
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    optimal_threshold = float(thresholds[best_idx])
    
    # Ensure threshold is sensible
    optimal_threshold = max(0.1, min(0.9, optimal_threshold))
    
    print(f"[CALIBRATION] Optimal Threshold: {optimal_threshold:.4f}")
        
    return optimal_threshold

def calibrate_model(model, val_loader, device):
    """
    Calibrates the model by fitting a temperature parameter on the validation set.
    """
    model.eval()
    nll_criterion = nn.BCEWithLogitsLoss()
    temperature_model = TemperatureScaling().to(device)
    
    # Collect all logits and labels from validation
    all_logits = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch[0], batch[1]
            images = images.to(device)
            logits = model(images)
            all_logits.append(logits)
            all_labels.append(labels.to(device).unsqueeze(1))
            
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    
    # Optimize temperature
    optimizer = torch.optim.LBFGS([temperature_model.temperature], lr=0.01, max_iter=50)
    
    def eval():
        optimizer.zero_grad()
        loss = nll_criterion(temperature_model(all_logits), all_labels)
        loss.backward()
        return loss
        
    optimizer.step(eval)
    
    temp = temperature_model.temperature.item()
    print(f"[CALIBRATION] Optimal Temperature found: {temp:.4f}")
    
    return temp


## Locked B-Run Orchestrator


In [ ]:
import os
import json
import time
import random
import shutil
import hashlib
import zipfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import confusion_matrix

RUN_ID = "B02_D_C0_seed42"
CONDITION = "D-C0"
ARCHITECTURE = "DenseNet121"
SEED = 42
USE_CBAM = False
USE_MASK_LOSS = False

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
RUNTIME_DIR = Path("/tmp/xai_locked_runtime") / RUN_ID
OUT_DIR = WORK_DIR / "locked_outputs" / "classification_runs" / CONDITION / ARCHITECTURE / f"seed_{SEED}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

DATASETS_REQUIRED = [
    "gate46-development-only-cxr",
    "gate7-test-seal",
    "cxr-lung-masks",
]


def setup_locked_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def sha256_file(path, chunk_size=1 << 20):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def audit_required_datasets():
    found = {name: len(list(INPUT_DIR.rglob(name))) > 0 for name in DATASETS_REQUIRED}
    missing = [name for name, ok in found.items() if not ok]
    audit = {"required": DATASETS_REQUIRED, "found": found, "missing": missing}
    if missing:
        raise FileNotFoundError(f"Missing required Kaggle datasets: {missing}")
    return audit


def environment_snapshot():
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "kaggle_input_children": sorted([p.name for p in INPUT_DIR.iterdir()]) if INPUT_DIR.exists() else [],
    }


def extract_archives_to_tmp():
    dev_blob_matches = list(INPUT_DIR.rglob("GATE46_DEVELOPMENT.g46blob"))
    if not dev_blob_matches:
        raise FileNotFoundError("GATE46_DEVELOPMENT.g46blob not found in Kaggle input.")
    dev_root = RUNTIME_DIR / "gate46_development_extracted"
    existing_dev = len(list(dev_root.rglob("*.jpg"))) + len(list(dev_root.rglob("*.jpeg"))) if dev_root.exists() else 0
    if existing_dev < 100:
        if dev_root.exists():
            shutil.rmtree(dev_root)
        dev_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(dev_blob_matches[0], "r") as zf:
            zf.extractall(dev_root)

    test_root = RUNTIME_DIR / "opaque_test_images_extracted"
    test_zip_matches = list(INPUT_DIR.rglob("opaque_test_images.zip"))
    if test_zip_matches:
        existing_test = len(list(test_root.rglob("*.jpg"))) + len(list(test_root.rglob("*.jpeg"))) if test_root.exists() else 0
        if existing_test < 100:
            if test_root.exists():
                shutil.rmtree(test_root)
            test_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(test_zip_matches[0], "r") as zf:
                zf.extractall(test_root)
    else:
        print("[AUDIT] opaque_test_images.zip not found; using images directly from /kaggle/input.")
    return dev_root, test_root


def label_values(df):
    if "label_index" in df.columns:
        return df["label_index"].astype(int).values
    if "label" in df.columns:
        if df["label"].dtype == object:
            return (df["label"].str.strip().str.upper() == "PNEUMONIA").astype(int).values
        return df["label"].astype(int).values
    raise KeyError("No label column found.")


def image_names_from_df(df):
    if "archive_member" in df.columns:
        return df["archive_member"].apply(lambda x: Path(str(x)).name).values
    if "source_relative_path" in df.columns:
        return df["source_relative_path"].apply(lambda x: Path(str(x)).name).values
    if "image" in df.columns:
        return df["image"].apply(lambda x: Path(str(x)).name).values
    if "image_id" in df.columns:
        return df["image_id"].apply(lambda x: Path(str(x)).name).values
    if "filename" in df.columns:
        return df["filename"].apply(lambda x: Path(str(x)).name).values
    raise KeyError("No image id/path column found.")


def load_manifests(dev_root):
    test_label_matches = list(INPUT_DIR.rglob("SEALED_test_label_key.csv"))
    if not test_label_matches:
        raise FileNotFoundError("SEALED_test_label_key.csv not found.")
    test_df = pd.read_csv(test_label_matches[0])

    dev_csvs = list(dev_root.rglob("*.csv")) + list(INPUT_DIR.rglob("*development*.csv"))
    dev_matches = [p for p in dev_csvs if "clean" in p.name.lower() and "development" in p.name.lower()]
    if not dev_matches:
        dev_matches = [p for p in dev_csvs if "development" in p.name.lower()]
    if not dev_matches:
        raise FileNotFoundError("Development manifest not found.")

    dev_df = pd.read_csv(dev_matches[0])
    if "final_split" in dev_df.columns:
        train_df = dev_df[dev_df["final_split"] == "train"].reset_index(drop=True)
        val_df = dev_df[dev_df["final_split"] == "tuning"].reset_index(drop=True)
    else:
        from sklearn.model_selection import train_test_split
        labels = label_values(dev_df)
        train_df, val_df = train_test_split(dev_df, test_size=0.15, random_state=3407, stratify=labels)
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)

    return train_df, val_df, test_df, dev_matches[0], test_label_matches[0]


def build_image_map(*roots):
    image_map = {}
    valid_exts = {".jpg", ".jpeg", ".png"}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in valid_exts and "mask" not in path.parent.name.lower():
                image_map[path.name] = path
    return image_map


def build_mask_map(*roots):
    mask_map = {}
    valid_exts = {".jpg", ".jpeg", ".png"}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in valid_exts:
                continue
            is_mask = "mask" in path.name.lower() or "mask" in path.parent.name.lower()
            if is_mask:
                mask_map[path.name] = path
    return mask_map


def find_mask_for_image(image_name, mask_map):
    stem = Path(image_name).stem
    candidates = [
        f"{stem}.png",
        f"{stem}.jpg",
        f"{stem}.jpeg",
        f"{stem}_mask.png",
        f"{stem}_lung_mask.png",
        f"{stem}_lungs.png",
    ]
    for candidate in candidates:
        if candidate in mask_map:
            return mask_map[candidate]
    stem_lower = stem.lower()
    for name, path in mask_map.items():
        normalized = Path(name).stem.lower().replace("_mask", "").replace("_lung", "").replace("_lungs", "")
        if normalized == stem_lower:
            return path
    return None


def make_canonical_mask_dir(all_image_names, mask_map):
    canonical_dir = RUNTIME_DIR / "lung_masks_canonical"
    if canonical_dir.exists():
        shutil.rmtree(canonical_dir)
    canonical_dir.mkdir(parents=True, exist_ok=True)
    coverage = {"total": len(all_image_names), "found": 0, "missing": []}
    for image_name in sorted(set(all_image_names)):
        mask_path = find_mask_for_image(image_name, mask_map)
        if mask_path is None:
            coverage["missing"].append(image_name)
            continue
        target = canonical_dir / image_name
        try:
            os.symlink(mask_path, target)
        except Exception:
            shutil.copy2(mask_path, target)
        coverage["found"] += 1
    coverage["missing_first_20"] = coverage["missing"][:20]
    coverage["missing_count"] = len(coverage["missing"])
    return canonical_dir, coverage


def expected_image_coverage(df, image_map):
    names = image_names_from_df(df)
    missing = [name for name in names if name not in image_map]
    return {"total": int(len(names)), "found": int(len(names) - len(missing)), "missing_count": int(len(missing)), "missing_first_20": missing[:20]}


def compute_ece(labels, probs, n_bins=15):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if hi == 1.0:
            mask = (probs >= lo) & (probs <= hi)
        if not np.any(mask):
            continue
        conf = probs[mask].mean()
        acc = labels[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def train_with_history(model, train_loader, val_loader, device, config, seed, class_weights=None):
    setup_locked_seed(seed)
    model = model.to(device)
    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = get_criterion(config, class_weights)
    scaler = GradScaler(device="cuda", enabled=config.mixed_precision and torch.cuda.is_available())
    history = []

    optimizer_stage1 = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config.stage1_lr, weight_decay=config.weight_decay)
    for epoch in range(config.stage1_epochs):
        current_mask_weight = config.mask_loss_weight * ((epoch + 1) / config.stage1_epochs) if config.use_mask_loss else 0.0
        train_loss, train_probs, train_labels = train_one_epoch(model, train_loader, optimizer_stage1, criterion, scaler, device, config, current_mask_weight)
        val_loss, val_probs, val_labels = validate(model, val_loader, criterion, device, config, current_mask_weight)
        val_metrics = calculate_metrics(val_labels.squeeze(), val_probs.squeeze(), threshold=0.5)
        row = {"stage": 1, "epoch": epoch + 1, "train_loss": float(train_loss), "val_loss": float(val_loss), "mask_weight": float(current_mask_weight)}
        row.update({f"val_{k}": v for k, v in val_metrics.items() if k != "confusion_matrix"})
        history.append(row)
        print(f"[{RUN_ID}] Stage 1 epoch {epoch+1}/{config.stage1_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auroc={val_metrics['auroc']:.4f}")

    model.unfreeze_all()
    optimizer_stage2 = AdamW(model.parameters(), lr=config.stage2_lr, weight_decay=config.weight_decay)
    scheduler = CosineAnnealingLR(optimizer_stage2, T_max=config.stage2_epochs, eta_min=config.stage2_min_lr)
    best_val_loss = float("inf")
    best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_epoch = 0
    patience_counter = 0

    for epoch in range(config.stage2_epochs):
        current_mask_weight = config.mask_loss_weight if config.use_mask_loss else 0.0
        train_loss, train_probs, train_labels = train_one_epoch(model, train_loader, optimizer_stage2, criterion, scaler, device, config, current_mask_weight)
        val_loss, val_probs, val_labels = validate(model, val_loader, criterion, device, config, current_mask_weight)
        scheduler.step()
        val_metrics = calculate_metrics(val_labels.squeeze(), val_probs.squeeze(), threshold=0.5)
        row = {"stage": 2, "epoch": epoch + 1, "train_loss": float(train_loss), "val_loss": float(val_loss), "lr": float(scheduler.get_last_lr()[0]), "mask_weight": float(current_mask_weight)}
        row.update({f"val_{k}": v for k, v in val_metrics.items() if k != "confusion_matrix"})
        history.append(row)
        print(f"[{RUN_ID}] Stage 2 epoch {epoch+1}/{config.stage2_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auroc={val_metrics['auroc']:.4f}")
        if val_loss < best_val_loss - config.min_delta:
            best_val_loss = float(val_loss)
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print(f"[{RUN_ID}] Early stopping at stage 2 epoch {epoch+1}.")
                break

    model.load_state_dict(best_weights)
    return model, pd.DataFrame(history), {"best_stage2_epoch": int(best_epoch), "best_val_loss": float(best_val_loss)}


def predict_with_logits(model, dataloader, device, config, temperature):
    model.eval()
    rows = []
    with torch.no_grad():
        for images, labels, names, masks in dataloader:
            images = images.to(device)
            labels_np = labels.cpu().numpy().astype(int)
            use_autocast = config.mixed_precision and torch.cuda.is_available()
            with autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", enabled=use_autocast):
                logits = model(images, return_attention=False).detach().cpu().numpy().reshape(-1)
                probs = torch.sigmoid(torch.tensor(logits / temperature)).numpy().reshape(-1)
                prob_list = [probs]
                logit_list = [logits]
                if config.use_tta:
                    images_hf = torch.flip(images, dims=[3])
                    logits_hf = model(images_hf, return_attention=False).detach().cpu().numpy().reshape(-1)
                    probs_hf = torch.sigmoid(torch.tensor(logits_hf / temperature)).numpy().reshape(-1)
                    prob_list.append(probs_hf)
                    logit_list.append(logits_hf)
                avg_probs = np.mean(prob_list, axis=0)
                avg_logits = np.mean(logit_list, axis=0)
            for name, label, raw_logit, prob in zip(names, labels_np, avg_logits, avg_probs):
                rows.append({"image_id": name, "true_label": int(label), "logit_raw": float(raw_logit), "temperature": float(temperature), "prob_pneumonia": float(prob)})
    return pd.DataFrame(rows)


def normalize_metrics(metrics, labels, probs, threshold, temperature):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs).astype(float)
    cm = metrics["confusion_matrix"]
    normalized = {
        "n_test": int(len(labels)),
        "n_normal": int((labels == 0).sum()),
        "n_pneumonia": int((labels == 1).sum()),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "accuracy": float(metrics["accuracy"]),
        "balanced_accuracy": float(metrics["balanced_accuracy"]),
        "sensitivity": float(metrics["sensitivity"]),
        "specificity": float(metrics["specificity"]),
        "precision": float(metrics["precision"]),
        "npv": float(cm["tn"] / (cm["tn"] + cm["fn"])) if (cm["tn"] + cm["fn"]) else 0.0,
        "f1": float(metrics["f1"]),
        "brier": float(metrics["brier_score"]),
        "ece": compute_ece(labels, probs, n_bins=15),
        "log_loss": float(metrics["log_loss"]),
        "mcc": float(metrics["mcc"]),
        "threshold": float(threshold),
        "temperature": float(temperature),
        "tn": int(cm["tn"]),
        "fp": int(cm["fp"]),
        "fn": int(cm["fn"]),
        "tp": int(cm["tp"]),
    }
    return normalized


def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2), encoding="utf-8")


run_started = time.time()
setup_locked_seed(SEED)
status = {"run_id": RUN_ID, "status": "STARTED", "started_unix": run_started}
save_json(OUT_DIR / "run_status.json", status)

try:
    presence_audit = audit_required_datasets()
    dev_root, test_root = extract_archives_to_tmp()
    train_df, val_df, test_df, dev_manifest_path, test_label_path = load_manifests(dev_root)
    image_map = build_image_map(INPUT_DIR, dev_root, test_root)
    mask_map = build_mask_map(INPUT_DIR, dev_root)

    all_names = list(image_names_from_df(train_df)) + list(image_names_from_df(val_df)) + list(image_names_from_df(test_df))
    canonical_mask_dir, mask_coverage = make_canonical_mask_dir(all_names, mask_map)

    dataset_audit = {
        "dataset_presence": presence_audit,
        "development_manifest": str(dev_manifest_path),
        "test_label_key": str(test_label_path),
        "n_train": int(len(train_df)),
        "n_val": int(len(val_df)),
        "n_test": int(len(test_df)),
        "train_label_counts": pd.Series(label_values(train_df)).value_counts().sort_index().astype(int).to_dict(),
        "val_label_counts": pd.Series(label_values(val_df)).value_counts().sort_index().astype(int).to_dict(),
        "test_label_counts": pd.Series(label_values(test_df)).value_counts().sort_index().astype(int).to_dict(),
        "image_map_size": int(len(image_map)),
        "train_image_coverage": expected_image_coverage(train_df, image_map),
        "val_image_coverage": expected_image_coverage(val_df, image_map),
        "test_image_coverage": expected_image_coverage(test_df, image_map),
        "mask_map_size": int(len(mask_map)),
        "canonical_mask_dir": str(canonical_mask_dir),
        "mask_coverage_all_splits": mask_coverage,
        "runtime_extraction_dir": str(RUNTIME_DIR),
    }
    save_json(OUT_DIR / "dataset_audit.json", dataset_audit)

    if dataset_audit["test_image_coverage"]["found"] != 624:
        raise RuntimeError(f"Expected 624 test images, found {dataset_audit['test_image_coverage']['found']}")
    if USE_MASK_LOSS and mask_coverage["missing_count"] > 0:
        raise RuntimeError(f"P run requires masks for all train/val/test images; missing {mask_coverage['missing_count']}")

    config = PipelineConfig()
    config.seeds = [SEED]
    config.models_to_train = [ARCHITECTURE]
    config.batch_size = 32
    config.num_workers = 2
    config.input_resolution = 256
    config.skip_training_if_checkpoint_exists = False
    config.use_cbam = USE_CBAM
    config.use_mask_loss = USE_MASK_LOSS
    config.apply_clahe = True
    config.loss_type = "Focal"
    config.use_tta = True
    config.mixed_precision = True
    config.stage1_epochs = 5
    config.stage2_epochs = 25
    config.patience = 5

    run_config = {
        "run_id": RUN_ID,
        "condition": CONDITION,
        "architecture": ARCHITECTURE,
        "seed": SEED,
        "use_cbam": USE_CBAM,
        "use_mask_loss": USE_MASK_LOSS,
        "config": config.__dict__,
        "dataset_slugs": DATASETS_REQUIRED,
        "protocol": "locked_final_battery_B",
    }
    save_json(OUT_DIR / "run_config.json", run_config)
    save_json(OUT_DIR / "environment.json", environment_snapshot())

    train_ds = ChestXRayDataset(train_df, image_map, config, split="train", mask_dir=canonical_mask_dir)
    val_ds = ChestXRayDataset(val_df, image_map, config, split="val", mask_dir=canonical_mask_dir)
    test_ds = ChestXRayDataset(test_df, image_map, config, split="test", mask_dir=canonical_mask_dir)
    train_loader, val_loader, test_loader = create_dataloaders(train_ds, val_ds, test_ds, config, use_sampler=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{RUN_ID}] Device: {device}")

    model = ClassificationModel(
        ARCHITECTURE,
        freeze_percent=config.densenet_freeze_percent if ARCHITECTURE == "DenseNet121" else config.resnet_freeze_percent,
        use_cbam=config.use_cbam,
    )
    class_weights = get_class_weights(train_df)
    model, history_df, training_summary = train_with_history(model, train_loader, val_loader, device, config, SEED, class_weights=class_weights)
    history_df.to_csv(OUT_DIR / "train_history.csv", index=False)

    temperature = calibrate_model(model, val_loader, device)
    _, val_probs, val_labels = validate(model, val_loader, get_criterion(config, class_weights.to(device)), device, config)
    val_probs_flat = val_probs.squeeze()
    val_labels_flat = val_labels.squeeze()
    threshold = find_optimal_threshold(val_labels_flat, val_probs_flat, metric="youden")
    save_json(OUT_DIR / "calibration.json", {"temperature": float(temperature), "method": "temperature_scaling_on_validation"})
    save_json(OUT_DIR / "threshold.json", {"threshold": float(threshold), "source": "validation_youden", "temperature": float(temperature)})

    pred_df = predict_with_logits(model, test_loader, device, config, temperature)
    pred_df["threshold"] = float(threshold)
    pred_df["pred_label"] = (pred_df["prob_pneumonia"] >= threshold).astype(int)
    pred_df["is_correct"] = pred_df["pred_label"] == pred_df["true_label"]
    pred_df["architecture"] = ARCHITECTURE
    pred_df["condition"] = CONDITION
    pred_df["seed"] = SEED
    pred_df["dataset_split"] = "sealed_test"
    pred_df["run_id"] = RUN_ID
    pred_df["logit_calibrated"] = np.log(np.clip(pred_df["prob_pneumonia"].values, 1e-7, 1 - 1e-7) / np.clip(1 - pred_df["prob_pneumonia"].values, 1e-7, 1 - 1e-7))
    pred_df["image_path_or_archive_member"] = pred_df["image_id"].map(lambda x: str(image_map[x]))
    pred_df["image_sha256"] = pred_df["image_path_or_archive_member"].map(lambda p: sha256_file(Path(p)))
    pred_df["mask_id"] = pred_df["image_id"]
    pred_df["mask_path"] = pred_df["image_id"].map(lambda x: str(canonical_mask_dir / x) if (canonical_mask_dir / x).exists() else "")
    pred_df["mask_sha256"] = pred_df["mask_path"].map(lambda p: sha256_file(Path(p)) if p else "")
    ordered_prediction_cols = [
        "image_id", "image_path_or_archive_member", "image_sha256",
        "mask_id", "mask_path", "mask_sha256", "true_label",
        "logit_raw", "temperature", "logit_calibrated", "prob_pneumonia",
        "threshold", "pred_label", "is_correct", "architecture", "condition",
        "seed", "dataset_split", "run_id",
    ]
    pred_df = pred_df[ordered_prediction_cols]
    pred_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)

    metrics_raw = calculate_metrics(pred_df["true_label"].values, pred_df["prob_pneumonia"].values, threshold=threshold)
    test_metrics = normalize_metrics(metrics_raw, pred_df["true_label"].values, pred_df["prob_pneumonia"].values, threshold, temperature)
    test_metrics.update({"run_id": RUN_ID, "architecture": ARCHITECTURE, "condition": CONDITION, "seed": SEED})
    save_json(OUT_DIR / "test_metrics.json", test_metrics)
    save_json(OUT_DIR / "confusion_matrix.json", {"tn": test_metrics["tn"], "fp": test_metrics["fp"], "fn": test_metrics["fn"], "tp": test_metrics["tp"]})
    torch.save(model.state_dict(), OUT_DIR / "best_checkpoint.pt")

    status = {
        "run_id": RUN_ID,
        "status": "SUCCESS",
        "runtime_seconds": float(time.time() - run_started),
        "n_test_predictions": int(len(pred_df)),
        "test_auroc": test_metrics["auroc"],
        "test_accuracy": test_metrics["accuracy"],
    }
    save_json(OUT_DIR / "run_status.json", status)
    print(json.dumps(status, indent=2))
    print(pd.DataFrame([test_metrics]))

except Exception as exc:
    import traceback
    status = {
        "run_id": RUN_ID,
        "status": "FAILED",
        "runtime_seconds": float(time.time() - run_started),
        "error": str(exc),
        "traceback": traceback.format_exc(),
    }
    save_json(OUT_DIR / "run_status.json", status)
    print(json.dumps(status, indent=2))
    raise
